# Skeleton Overlay

Demonstrates `kinema.render.overlay` — draws pose skeleton and optional COM trail on each video frame.

**Contents**
1. POSE_CONNECTIONS — visualise the 35 BlazePose skeleton edges on a stick figure
2. Load keypoints + video metadata
3. Render skeleton-only overlay
4. Render with COM trail
5. Inline QA — sample frames from both outputs
6. QA checks — frame count, dimensions

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from kinema.io.keypoints import read_keypoints, LANDMARK_NAMES
from kinema.io.video import probe_video
from kinema.render.overlay import POSE_CONNECTIONS, render_overlay

RAW_DIR   = Path('../data/raw')
DATA_DIR  = Path('../data/processed')
STEM      = 'VID-20260528-WA0044'

VIDEO_PATH     = RAW_DIR  / f'{STEM}.mp4'
KP_PATH        = DATA_DIR / f'{STEM}_keypoints.parquet'
COM_PATH       = DATA_DIR / f'{STEM}_com.parquet'
OUT_SKEL       = DATA_DIR / f'{STEM}_overlay.mp4'
OUT_COM        = DATA_DIR / f'{STEM}_overlay_com.mp4'

meta = probe_video(VIDEO_PATH)
print(f'Video  : {meta.width}x{meta.height}  {meta.fps:.2f} fps  {meta.frame_count} frames  {meta.duration_sec:.1f}s')

## 1. POSE_CONNECTIONS — skeleton topology

35 edges connecting the 33 BlazePose landmarks.  
Landmark indices follow MediaPipe's standard ordering (NOSE=0 … RIGHT_FOOT_INDEX=32).

In [ ]:
# Approximate 2-D positions for a standing stick figure (x right, y up)
_STICK: dict[int, tuple[float, float]] = {
    # face
    0:  ( 0.00,  1.70),   # NOSE
    1:  (-0.03,  1.73),   # LEFT_EYE_INNER
    2:  (-0.06,  1.74),   # LEFT_EYE
    3:  (-0.09,  1.73),   # LEFT_EYE_OUTER
    4:  ( 0.03,  1.73),   # RIGHT_EYE_INNER
    5:  ( 0.06,  1.74),   # RIGHT_EYE
    6:  ( 0.09,  1.73),   # RIGHT_EYE_OUTER
    7:  (-0.12,  1.68),   # LEFT_EAR
    8:  ( 0.12,  1.68),   # RIGHT_EAR
    9:  (-0.04,  1.63),   # MOUTH_LEFT
    10: ( 0.04,  1.63),   # MOUTH_RIGHT
    # torso
    11: (-0.20,  1.40),   # LEFT_SHOULDER
    12: ( 0.20,  1.40),   # RIGHT_SHOULDER
    23: (-0.15,  0.90),   # LEFT_HIP
    24: ( 0.15,  0.90),   # RIGHT_HIP
    # left arm
    13: (-0.40,  1.20),   # LEFT_ELBOW
    15: (-0.55,  0.95),   # LEFT_WRIST
    17: (-0.60,  0.88),   # LEFT_PINKY
    19: (-0.58,  0.87),   # LEFT_INDEX
    21: (-0.56,  0.89),   # LEFT_THUMB
    # right arm
    14: ( 0.40,  1.20),   # RIGHT_ELBOW
    16: ( 0.55,  0.95),   # RIGHT_WRIST
    18: ( 0.60,  0.88),   # RIGHT_PINKY
    20: ( 0.58,  0.87),   # RIGHT_INDEX
    22: ( 0.56,  0.89),   # RIGHT_THUMB
    # left leg
    25: (-0.18,  0.50),   # LEFT_KNEE
    27: (-0.20,  0.10),   # LEFT_ANKLE
    29: (-0.22, -0.02),   # LEFT_HEEL
    31: (-0.16, -0.03),   # LEFT_FOOT_INDEX
    # right leg
    26: ( 0.18,  0.50),   # RIGHT_KNEE
    28: ( 0.20,  0.10),   # RIGHT_ANKLE
    30: ( 0.22, -0.02),   # RIGHT_HEEL
    32: ( 0.16, -0.03),   # RIGHT_FOOT_INDEX
}

fig, ax = plt.subplots(figsize=(5, 9))

for a, b in POSE_CONNECTIONS:
    if a in _STICK and b in _STICK:
        xa, ya = _STICK[a]
        xb, yb = _STICK[b]
        ax.plot([xa, xb], [ya, yb], color='steelblue', lw=2, zorder=1)

for lid, (x, y) in _STICK.items():
    ax.scatter(x, y, color='tomato', s=40, zorder=2)
    ax.annotate(str(lid), (x, y), fontsize=6, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')

ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'POSE_CONNECTIONS  ({len(POSE_CONNECTIONS)} edges, 33 landmarks)', fontsize=11)
plt.tight_layout()
plt.show()

print(f'Total edges : {len(POSE_CONNECTIONS)}')
print(f'Total landmarks: {len(LANDMARK_NAMES)}')

## 2. Load keypoints

In [ ]:
kp = read_keypoints(KP_PATH)

n_frames     = kp['frame_idx'].nunique()
n_rows       = len(kp)
vis_pct      = (kp['visibility'] > 0.5).mean() * 100

print(f'Keypoints file : {KP_PATH.name}')
print(f'Frames with keypoints : {n_frames}')
print(f'Total rows            : {n_rows}  ({n_rows / n_frames:.1f} landmarks/frame avg)')
print(f'Visibility > 0.5      : {vis_pct:.1f}%')

kp.head()

## 3. Render skeleton-only overlay

Cyan skeleton, no COM marker.  
Writes `{STEM}_overlay.mp4` with the same fps and resolution as the source.

In [ ]:
render_overlay(
    VIDEO_PATH,
    KP_PATH,
    OUT_SKEL,
    visibility_threshold=0.5,
)
print(f'Written → {OUT_SKEL}  ({OUT_SKEL.stat().st_size / 1e6:.1f} MB)')

## 4. Render with COM trail

Adds a magenta marker at the centre-of-mass position and a 1-second fading trail.

In [ ]:
render_overlay(
    VIDEO_PATH,
    KP_PATH,
    OUT_COM,
    com_path=COM_PATH,
    visibility_threshold=0.5,
)
print(f'Written → {OUT_COM}  ({OUT_COM.stat().st_size / 1e6:.1f} MB)')

## 5. Inline QA — sample frames

Pull four evenly spaced frames from each output and display side-by-side.

In [ ]:
def _sample_frames(path: Path, n: int = 4) -> list[np.ndarray]:
    """Read n evenly-spaced frames from a video file (RGB)."""
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


skel_frames = _sample_frames(OUT_SKEL)
com_frames  = _sample_frames(OUT_COM)

n = len(skel_frames)
fig, axes = plt.subplots(2, n, figsize=(4 * n, 5))

labels = ['Skeleton only', 'Skeleton + COM trail']
for row, (frames, label) in enumerate(zip([skel_frames, com_frames], labels)):
    for col, frame in enumerate(frames):
        ax = axes[row][col]
        ax.imshow(frame)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(label, fontsize=9, labelpad=6)

fig.suptitle('Sample overlay frames', fontsize=12)
plt.tight_layout()
plt.show()

## 6. QA checks — frame count and dimensions

In [ ]:
def _video_stats(path: Path) -> dict:
    cap = cv2.VideoCapture(str(path))
    n = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        n += 1
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {'frames': n, 'width': w, 'height': h}


skel_stats = _video_stats(OUT_SKEL)
com_stats  = _video_stats(OUT_COM)

print(f'Source  : {meta.frame_count} frames  {meta.width}x{meta.height}')
print(f'Skel    : {skel_stats["frames"]} frames  {skel_stats["width"]}x{skel_stats["height"]}')
print(f'COM     : {com_stats["frames"]} frames  {com_stats["width"]}x{com_stats["height"]}')

assert skel_stats['frames'] == meta.frame_count, 'Frame count mismatch (skeleton)'
assert com_stats['frames']  == meta.frame_count, 'Frame count mismatch (COM)'
assert skel_stats['width']  == meta.width,       'Width mismatch'
assert skel_stats['height'] == meta.height,      'Height mismatch'

print('\nAll QA checks passed.')